# Adjustment Test Harness

Runs any adjustment scenario end to end: preview, submit, wait for the pipeline,
verify the numbers actually landed, clean up.

**How to use it**

1. Run **setup**, then edit **config** and run it.
2. Run **preflight**. Fix anything it reports before going further.
3. Run **seed** once per environment. The reserved test COB holds no fact data,
   so without this every preview returns zero rows and verifies nothing.
4. Then run whichever mode you want: `single`, `smoke`, `full`, `negative`.

**Safety.** Everything writes to COB 20991231 by default. Pointing at a real COB
needs both `cob` and `allow_real_cob` set, and seeding refuses to run anywhere
but the test COB.

Design: `docs/superpowers/specs/2026-09-23-adjustment-test-harness-design.md`

In [ ]:
from snowflake.snowpark.context import get_active_session
import adjustment_test_kit as kit

session = get_active_session()
scopes  = kit.load_scopes(session)

def show(r):
    """Print one scenario result: preview numbers, every check, timings."""
    print(("PASS  " if r.passed else "FAIL  ") + r.scenario.id +
          "  |  " + r.scenario.scope + "  " + r.scenario.adjustment_type)
    if r.preview:
        print("  preview   rows=%s  current=%s  delta=%s  projected=%s"
              % (r.preview.get("rows"), r.preview.get("current"),
                 r.preview.get("delta"), r.preview.get("projected")))
    print("  adjustment %s   status %s" % (r.adj_id or "-", r.status or "-"))
    if r.message:
        print("  message   " + r.message[:200])
    for name, ok, detail in r.checks:
        print("    " + ("ok   " if ok else "FAIL ") + name +
              (("  -  " + detail) if detail else ""))
    if r.pruning.get("checked"):
        print("  pruning   " + r.pruning.get("detail", ""))
    print("  timings   %s   total %ss" % (r.timings, r.elapsed))
    print("  cleaned   %s" % r.cleaned)
    print()

def show_all(results):
    for r in results:
        show(r)
    passed = sum(1 for r in results if r.passed)
    print("=" * 60)
    print("%d of %d scenarios passed" % (passed, len(results)))

print("Scopes configured:")
for _name, _sc in scopes.items():
    print("  %-12s %-9s fact=%-42s measure=%s"
          % (_name, "active" if _sc.is_active else "INACTIVE",
             _sc.fact_table, _sc.metric))

## Configuration

Everything you change lives in the next cell. Re-run it after any edit.

In [ ]:
cfg = kit.HarnessConfig(
    # ── Where the test writes ────────────────────────────────────────────
    cob              = kit.TEST_COB,   # 20991231, the reserved test COB
    allow_real_cob   = False,          # must be True to point at a real COB

    # ── Seeding ──────────────────────────────────────────────────────────
    # Real fact rows are cloned from this COB into the test COB, changing
    # only the COB. Set it to a COB that actually holds data for your books.
    seed_source_cob  = 20260420,
    seed_rows        = 2000,

    # ── What the scenarios act on ────────────────────────────────────────
    book_code        = "UATBOOK",      # source book, and the book most types filter on
    target_book_code = "UATBOOK2",     # transfer target
    entity_code      = "",             # blank resolves from book_code below

    # ── Pipeline ─────────────────────────────────────────────────────────
    # force=True calls the escape-hatch procedure instead of waiting for the
    # one-minute task. Faster, but it does not exercise the real queue.
    force            = False,
    timeout_s        = 900,
    poll_s           = 10,

    # ── Behaviour ────────────────────────────────────────────────────────
    keep_failures    = True,           # leave a failed run's rows for inspection
    max_scan_ratio   = 0.80,           # pruning guard threshold
)

if not cfg.entity_code:
    _info = kit.resolve_book(session, cfg.book_code)
    cfg.entity_code = (_info or {}).get("entity_code") or ""
    print("entity resolved from %s -> %s" % (cfg.book_code, cfg.entity_code or "NOT FOUND"))

cfg.validate()
print("Config valid. COB %s, %s."
      % (cfg.cob, "forced processing" if cfg.force else "waiting on the pipeline tasks"))

## Pre-flight

Checks the environment before anything is submitted. The important one is
whether the pipeline tasks are started: without them a run just waits for the
timeout with no explanation.

In [ ]:
kit.ensure_bots(session, cfg)
problems = kit.preflight(session, cfg, scopes)

if problems:
    print("PRE-FLIGHT PROBLEMS — fix these before running a scenario:")
    for _p in problems:
        print("  - " + _p)
else:
    print("Pre-flight clean.")

## Seed the test COB

Run this once per environment, and again whenever you change the books. It
clones a pinned slice of real fact rows into the test COB, changing only the
COB, so every key and measure stays consistent with real data.

It is idempotent: a scope that already has rows at the test COB is skipped.

In [ ]:
_keys = []
for _code in (cfg.book_code, cfg.target_book_code):
    _info = kit.resolve_book(session, _code)
    if _info and _info.get("book_key") is not None:
        _keys.append(_info["book_key"])
        if _info.get("ambiguous"):
            print("WARNING: book %s maps to more than one entity" % _code)
    else:
        print("WARNING: book %s not found in DIMENSION.BOOK" % _code)

if not _keys:
    print("No book keys resolved. Fix book_code / target_book_code in config.")
else:
    for _name, _sc in scopes.items():
        if not _sc.is_active:
            continue
        try:
            _n = kit.seed(session, _sc, cfg, book_keys=_keys)
            _have = kit.count_fact_rows(session, _sc, cfg.cob)
            print("%-12s inserted=%-7s  rows now at COB %s: %s"
                  % (_name, _n, cfg.cob, _have))
        except Exception as _exc:
            print("%-12s skipped: %s" % (_name, _exc))

## Single scenario

Pick a scope and a type, set the parameters, run it. This is the everyday use.
The widget cell is optional: you can equally just edit the scenario below.

In [ ]:
import streamlit as st

_active = [n for n, s in scopes.items() if s.is_active]
SCOPE  = st.selectbox("Scope", _active)
TYPE   = st.selectbox("Adjustment type", list(kit.ALL_TYPES), index=1)
FACTOR = st.number_input("Scale factor (Scale, Roll, Transfer)",
                         value=1.05, step=0.01, format="%.4f")
st.caption("Run the next cell to execute this scenario.")

In [ ]:
scenario = kit.Scenario(
    id              = "MANUAL-01",
    scope           = SCOPE,
    adjustment_type = TYPE,
    params          = {
        # Filters. Which of these apply depends on the scope; see
        # kit.SCOPE_FILTER_FIELDS[SCOPE] for the full list.
        "book_code":    cfg.book_code,
        "scale_factor": FACTOR,
    },
    keep = False,      # True leaves the adjustment in place for inspection
)

# Transfer and Entity Roll need different parameters, so fill them in here.
if TYPE == "Transfer":
    scenario.params = {"source_book_code": cfg.book_code,
                       "book_code": cfg.target_book_code,
                       "scale_factor": FACTOR}
elif TYPE == "EROL":
    scenario.params = {"entity_code": cfg.entity_code,
                       "source_cobid": cfg.seed_source_cob}
elif TYPE == "Roll":
    scenario.params["source_cobid"] = cfg.seed_source_cob
elif TYPE == "Flatten":
    scenario.params.pop("scale_factor", None)

print("Filters that apply to %s: %s"
      % (SCOPE, ", ".join(sorted(kit.SCOPE_FILTER_FIELDS.get(SCOPE, [])))))
print()

result = kit.run_scenario(session, scenario, cfg, scopes=scopes)
show(result)

## Smoke

One representative Scale per active scope. The fast post-deploy sanity check.

In [ ]:
smoke_scenarios = kit.build_matrix(scopes, "smoke", cfg)
print("Running %d scenarios.\n" % len(smoke_scenarios))
smoke_results = kit.run_suite(session, smoke_scenarios, cfg, scopes=scopes)
show_all(smoke_results)

## Full matrix

Every legal scope by type combination. This submits real work and waits on the
pipeline for each one, so expect it to take a while. Scenarios run in order
because the engine serialises combinations that write the same table.

In [ ]:
full_scenarios = kit.build_matrix(scopes, "full", cfg)
print("Running %d scenarios. At roughly a minute each this is a long run.\n"
      % len(full_scenarios))

full_results = kit.run_suite(session, full_scenarios, cfg, scopes=scopes,
                             on_result=show)
print("=" * 60)
print("%d of %d scenarios passed"
      % (sum(1 for r in full_results if r.passed), len(full_results)))

## Negative tests

The submit procedure's hard rejections, one scenario each. These never reach
the pipeline, so they are fast and safe to run anywhere.

In [ ]:
negative_scenarios = kit.build_matrix(scopes, "negative", cfg)
negative_results = kit.run_suite(session, negative_scenarios, cfg, scopes=scopes)
show_all(negative_results)

## Leak detector and concurrency check

Run the leak detector after a full run: it asserts the test COB is empty again.
The concurrency check submits two adjustments that write the same table and
asserts their processing windows did not overlap.

In [ ]:
leaks = kit.check_leaks(session, cfg, scopes)
if leaks:
    print("Rows left behind at COB %s:" % cfg.cob)
    for _what, _n in leaks:
        print("  %-55s %s" % (_what, _n))
    print()
    print("Seeded fact rows are expected here if you want to keep the COB")
    print("populated. Remove them with kit.unseed(session, scopes['VaR']).")
else:
    print("No leaks. The test COB is clean.")

In [ ]:
_scope = [n for n, s in scopes.items() if s.is_active][0]
_pair = [
    kit.Scenario("CONC-01", _scope, "Scale",
                 {"book_code": cfg.book_code, "scale_factor": 1.01}, keep=True),
    kit.Scenario("CONC-02", _scope, "Scale",
                 {"book_code": cfg.book_code, "scale_factor": 1.02}, keep=True),
]
_conc = kit.run_suite(session, _pair, cfg, scopes=scopes)
show_all(_conc)

_verdict = kit.check_serialisation(session, [r.adj_id for r in _conc if r.adj_id])
print("Serialisation:", _verdict.get("detail"))

for _r in _conc:
    if _r.adj_id:
        kit.cleanup(session, _r.adj_id, scopes.get(_r.scenario.scope), cfg)
print("Cleaned up both.")

## Report

Markdown summary in the same shape as the existing UAT report, including the
per-phase timings. Copy it into a document, or paste it into a pull request.

In [ ]:
# Point this at whichever result list you want to report on.
results_to_report = globals().get("full_results") \
    or globals().get("smoke_results") \
    or globals().get("negative_results") \
    or []

if not results_to_report:
    print("Run one of the modes above first.")
else:
    report_md = kit.render_report(results_to_report, cfg,
                                  title="Adjustment harness run")
    try:
        import streamlit as st
        st.markdown(report_md)
    except Exception:
        print(report_md)